### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
# Read all the pdf's inside the directory 

def process_all_pdf(pdf_directory):
    """Process all the PDF files"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f' loaded {len(documents)} pages')

        except Exception as e:
            print(f' Error:{e}')

    print(f'\nTotal documents loaded: {len(all_documents)}')
    return all_documents 

all_pdf_documents = process_all_pdf("../data")

Found 3 PDF files to process

Processing: AI Agents.pdf
 loaded 30 pages

Processing: Claude-Code.pdf
 loaded 23 pages

Processing: Anthropic.pdf
 loaded 2 pages

Total documents loaded: 55


In [ ]:
all_pdf_documents

In [10]:
### Text Splitting into chunks 

def split_documents(documents,chunk_size = 1000, chunk_overlap= 200):
    """Splitting documents into smaller chunks for RAG Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')

    # Example of a Chunk 
    if split_docs:
        print(f'\nExample Chunk:')
        print(f'\ncontent: {split_docs[0].page_content[:200]}...')
        print(f'\nMetadata: {split_docs[0].metadata}')

    return split_docs 


In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

### Embedding and vector store DB 

In [12]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/durgaprasadreddypralayakaveri/Documents/GitHub/RAG-LangChain-Pipeline/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = 'all-MiniLM-l6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:    
            print(f'Error Loading model {self.model_name}: {e}')
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f'Generating embeddings for {len(texts)} texts....')
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f'Generated embeddings with shape: {embeddings.shape}')
        return embeddings 
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-l6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5144.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


/var/folders/x2/smsrx5p17hd6dqglw21wmzmm0000gn/T/ipykernel_33903/3489931333.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store 

In [14]:
class VectorStore:
    """Manages embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f'vector store intialized. collection: {self.collection_name}')
            print(f'Existing documents in collection: {self.collection.count()}')

        except Exception as e:
            print(f'Error intializing vector store: {e}')
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        
        if len(documents) != len(embeddings):
            raise ValueError(" Number of documents must match embeddings")
        
        print(f'Adding {len(documents)} documents to vector store')

        # Data for Chromadb 
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            try: 
                self.collection.add(
                    ids=ids,
                    embeddings = embeddings_list,
                    metadatas = metadatas,
                    documents = documents_text
                ) 
                print(f'Successfully added {len(documents)} documents to vector store')
                print(f'Total documents in collection: {self.collection.count()}')

            except Exception as e:
                print(f'Error adding docuemnts to vector store: {e}')
                raise 

vector_store = VectorStore()
vector_store

vector store intialized. collection: pdf_documents
Existing documents in collection: 0


In [18]:
# convert text to embeddings 
texts = [doc.page_content for doc in chunks]


In [ ]:
# Generate Embeddings 

embeddings = embedding_manager.generate_embeddings(texts)

# store in vector database 

vector_store.add_documents(chunks, embeddings)

### Retriever Pipeline From VectorStore

In [27]:
class RAGRetriever:
    """Handles Query based Retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0)-> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        # Generating Query Embeddings 
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1 
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filter)")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [29]:
rag_retriever.retrieve("What topics are covered in the Anthropic document")

Retrieving documents for query: 'What topics are covered in the Anthropic document'
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filter)


[{'id': 'doc_d2706f54_233',
  'content': 'including regularly reviewing documentation, guides, and \ncode samples at docs.anthropic.com.\nPrepare for the Future: Progressing with \nimplementation is not a one-time task. It takes time to \niterate and improve. Anthropic is at the forefront of AI \ndevelopment, consistently pushing the boundaries of \nwhat’s possible. Partnering with Anthropic means you can \nbuild more confidently with frontier AI you can trust to be \nsafer, more secure, and more reliable.',
  'metadata': {'page': 1,
   'creationdate': '2025-04-01T18:39:35-04:00',
   'context_length': 461,
   'total_pages': 2,
   'producer': 'Adobe PDF Library 17.0',
   'source': '../data/text_files/pdf/Anthropic.pdf',
   'page_label': '2',
   'trapped': '/False',
   'file_type': 'pdf',
   'source_file': 'Anthropic.pdf',
   'doc_index': 233,
   'moddate': '2025-04-01T18:39:35-04:00',
   'creator': 'Adobe InDesign 20.2 (Macintosh)'},
  'similarity_score': 0.1518673300743103,
  'distance